In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from dotenv import load_dotenv
import os
import snowflake.connector
import getpass

In [2]:
load_dotenv(override=True)
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    password=getpass.getpass('Snowflake Password: '),
    role=os.getenv('SNOWFLAKE_ROLE'),
    warehouse='COMPUTE_WH',
    database='OLIST_ECOMMERCE',
    schema='ANALYTICS'
)

Snowflake Password:  ········


In [3]:
query = "SELECT * FROM INT_ORDER_ITEMS_TRANSLATED"
df_items = pd.read_sql(query, conn)
conn.close()

C:\Users\awang\AppData\Local\Temp\ipykernel_14088\2740487904.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_items = pd.read_sql(query, conn)


In [4]:
basket = (df_items
          .groupby(['ORDER_ID', 'PRODUCT_CATEGORY'])['PRODUCT_CATEGORY']
          .count().unstack().reset_index().fillna(0)
          .set_index('ORDER_ID'))

In [5]:
basket_sets = basket.map(lambda x: 1 if x > 0 else 0)

In [6]:
frequent_itemsets = apriori(basket_sets, min_support=0.0005, use_colnames=True)

C:\Users\awang\Documents\Analytic_project\olist_ecommerce\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [7]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

In [8]:
top_rules = rules.sort_values('lift', ascending=False).head(10)

In [9]:
print("TOP PRODUCT ASSOCIATIONS")
display(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

TOP PRODUCT ASSOCIATIONS


,antecedents,consequents,support,confidence,lift


In [10]:
basket_sizes = basket_sets.sum(axis=1)
multi_item_baskets = basket_sizes[basket_sizes > 1].count()
print(f"Total Orders: {len(basket_sizes):,}")
print(f"Orders with multiple different categories: {multi_item_baskets:,}")

Total Orders: 97,277
Orders with multiple different categories: 727


In [12]:
if not frequent_itemsets.empty:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.5)
    top_rules = rules.sort_values('lift', ascending=False).head(10)

    print("TOP PRODUCT ASSOCIATIONS")
    display(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
else:
    print("Even with lowered thresholds, the data is too sparse to find frequent itemsets.")

TOP PRODUCT ASSOCIATIONS


,antecedents,consequents,support,confidence,lift
0,(home_confort),(bed_bath_table),0.000442,0.108312,1.118859
1,(bed_bath_table),(home_confort),0.000442,0.004566,1.118859
2,(construction_tools_lights),(furniture_decor),0.000113,0.045082,0.680018
3,(furniture_decor),(construction_tools_lights),0.000113,0.001706,0.680018


### Key Business Findings:

**1. The "Cross-Sell" Opportunity (Lift > 1)**
* **Association:** `Home Comfort` → `Bed Bath Table`
* **Lift:** `1.118`
* **Strategic Action:** Customers buying home comfort items are **1.11x more likely** to also purchase bed and bath products than random chance would dictate. Olist should implement a "Frequently Bought Together" widget specifically pairing these categories on the product checkout pages to increase Average Order Value (AOV).

**2. The "Anti-Pattern" (Lift < 1)**
* **Association:** `Construction Tools & Lights` → `Furniture Decor`
* **Lift:** `0.680`
* **Strategic Action:** These items actively repel each other. Customers buying construction lighting are significantly *less* likely to buy furniture decor simultaneously. Marketing spend should not be wasted on cross-promoting these specific categories together.